In [ ]:
!pip install transformers datasets accelerate wandb peft sentencepiece bitsandbytes
!pip install -U datasets
!pip install tf-keras
!pip install --upgrade "jinja2>=3.1.0"

In [1]:
import psutil

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

from transformers.cache_utils import DynamicCache

# A patch to make DeepSeek work on current transformers version without downgrading it
if not hasattr(DynamicCache, 'get_max_length'):
    DynamicCache.get_max_length = lambda self: None  # or a large number (e.g., 1000000)


from datasets import load_dataset, DatasetDict, Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          PreTrainedTokenizer, PreTrainedModel, TrainingArguments,
                          DataCollatorForLanguageModeling, BitsAndBytesConfig, PreTrainedTokenizerBase
                          )

from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model



import torch
torch.set_float32_matmul_precision("high")
import gc
from transformers import DataCollatorWithPadding
import math

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-21 22:16:44.775455: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753136204.785376   70105 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753136204.790136   70105 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753136204.795903   70105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00

In [2]:
ENABLE_QUANTIZATION_FOR_LORA = True
print(f'QUANTIZATION HAS BEEN {"ENABLED" if ENABLE_QUANTIZATION_FOR_LORA else "DISABLED"}')

QUANTIZATION HAS BEEN ENABLED


In [3]:
def print_mem_usage():
    process = psutil.Process()
    ram_used = process.memory_info().rss / 1000**2

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1000**2
        reserved = torch.cuda.memory_reserved() / 1000**2
        total = torch.cuda.get_device_properties(0).total_memory / 1000**2
        free = reserved - allocated

        print(f"CPU RAM Used       : {ram_used:.2f} MB")
        print(f"GPU VRAM Allocated : {allocated:.2f} MB")
        print(f"GPU VRAM Reserved  : {reserved:.2f} MB")
        print(f"GPU VRAM Free (torch): {free:.2f} MB")
        print(f"GPU VRAM Total     : {total:.2f} MB")
    else:
        print(f"CPU RAM Used       : {ram_used:.2f} MB")
        print("GPU not available.")

In [4]:
import random
import numpy as np
import torch
import os

def set_seed(seed: int = 42):
    random.seed(seed)  # Python RNG
    np.random.seed(seed)  # NumPy RNG
    torch.manual_seed(seed)  # PyTorch CPU RNG
    torch.cuda.manual_seed(seed)  # PyTorch current GPU RNG
    torch.cuda.manual_seed_all(seed)  # All GPUs
    torch.backends.cudnn.deterministic = True  # Makes results deterministic
    torch.backends.cudnn.benchmark = False  # Disables autotuner that could introduce randomness
    os.environ["PYTHONHASHSEED"] = str(seed)  # Python hashing (used in dicts, sets, etc.)

set_seed(42)

In [5]:
def load_ds(path: str = "FINAL_p4_ds_clean_comments.jsonl"):
  dataset = load_dataset("json", data_files=path)
  return dataset

In [6]:
def train_val_test_split(dataset: Dataset, train_size: float = 0.85, val_size: float = 0.05, test_size: float = 0.10):
  assert train_size + val_size + test_size == 1.0

  X = dataset.train_test_split(train_size=train_size)

  X2 = X["test"].train_test_split(train_size = (val_size / (1 - train_size)) )

  return DatasetDict(
    {
      "train": X["train"],
      "validation": X2["train"],
      "test": X2["test"]
    }
  )

In [7]:
from transformers import AutoConfig

class ModelLoader:
  _instance = {}

  def __init__(self):
    raise RuntimeError("This is a singleton class! Use get_instance(checkpoint: str) method instead!")

  # "bigcode/starcoder2-15b-instruct-v0.1"
  # "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
  @classmethod
  def get_instance(cls, checkpoint: str = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"):
    if checkpoint in cls._instance:
      return cls._instance[checkpoint]

    config = AutoConfig.from_pretrained(checkpoint, trust_remote_code=True)

    # IMPORTANT: attention dropout is disabled to work with flash-attention-3. If you're not using it, you can enable it.
    # config.attention_dropout = 0.0

    # if you have set the ENABLE_QUANTIZATION_FOR_LORA=False,
    # model would be loaded without this quantization config giving you a LoRA setup, not QLoRA
    if ENABLE_QUANTIZATION_FOR_LORA:
        nf4_config = BitsAndBytesConfig(
          load_in_4bit=True,
          bnb_4bit_quant_type="nf4",
          bnb_4bit_use_double_quant=True,
          bnb_4bit_compute_dtype=torch.bfloat16
        )

    # enable attn_implementation = "flash_attention_3" if you have it installed.
    model = AutoModelForCausalLM.from_pretrained(
      checkpoint,
      quantization_config=nf4_config if ENABLE_QUANTIZATION_FOR_LORA else None,
      torch_dtype=torch.bfloat16,
      trust_remote_code=True,
      device_map="auto",
      attn_implementation = "flash_attention_2",
      config=config
    )

    model.config.use_cache = False
    model.gradient_checkpointing_disable()

    tokenizer = AutoTokenizer.from_pretrained(checkpoint, trust_remote_code=True)

    tokenizer.padding_side   = "right"
    tokenizer.truncation_side = "right"

    cls._instance[checkpoint] = {"model": model, "tokenizer": tokenizer}

    return cls._instance[checkpoint]

  # "bigcode/starcoder2-15b-instruct-v0.1"
  # "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
  @classmethod
  def delete_instance(cls, checkpoint: str = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"):
      if checkpoint not in cls._instance:
          return

      instance = cls._instance[checkpoint]

      model = instance.get("model")

      if model is not None:
          model.cpu()
          for attr in dir(model):
              try:
                  delattr(model, attr)
              except:
                  pass
          del model


      tokenizer = instance.get("tokenizer")
      if tokenizer is not None:
          del tokenizer

      del cls._instance[checkpoint]

      gc.collect()
      torch.cuda.empty_cache()


In [8]:
def add_special_tokens_to_tokenizer(tokenizer: PreTrainedTokenizerBase, model: AutoModelForCausalLM, new_tokens=["<p4>", "</p4>"]):
  existing = tokenizer.additional_special_tokens
  combined_special_tokens = list(set(existing + new_tokens))
  tokenizer.add_special_tokens({"additional_special_tokens": combined_special_tokens})

  # add special padding token if needed for collation
  if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

  emb_size_mltpl_32 = math.ceil(len(tokenizer) / 32) * 32
  model.resize_token_embeddings(emb_size_mltpl_32)

In [9]:


def prepare_dataloaders(dataset_splits: DatasetDict, tokenizer, batch_size=8):
  ret = {}

  collate_fn = DataCollatorWithPadding(tokenizer, padding=True) #padding=True, 'max_length'

  for split in dataset_splits:
    dataloader = torch.utils.data.DataLoader(
        dataset=dataset_splits[split],
        collate_fn=collate_fn,
        batch_size=batch_size,
        pin_memory=True,
        num_workers=os.cpu_count()
      )

    ret[split] = dataloader

  return ret

In [10]:
# training loop without trainer:  https://discuss.huggingface.co/t/training-loop-for-lora/106885

# how to prep model for qlora: https://huggingface.co/docs/peft/en/developer_guides/quantization
# https://huggingface.co/docs/peft/task_guides/prompt_based_methods

def prepare_model_for_qlora(model, lora_config=None):
  if hasattr(model, "peft_config"):
    raise RuntimeError("Model already has peft config, delete it and run again!")

  lora_config = LoraConfig(
    r = 32,
    lora_alpha = 64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
  )

  # if you have set the ENABLE_QUANTIZATION_FOR_LORA=False,
  # you would be loading this model in its full size which would make it LoRA, not QLoRA
  if ENABLE_QUANTIZATION_FOR_LORA:
      model = prepare_model_for_kbit_training(model)

  model = get_peft_model(model, lora_config)

  model.print_trainable_parameters()

  return model

In [11]:
# ds = train_val_test_split(load_ds()["train"], train_size=0.75, test_size=0.125, val_size=0.125)
ds = train_val_test_split(load_ds()["train"], train_size=0.75, test_size=0.125, val_size=0.125)

model, tokenizer = ModelLoader.get_instance().values()

add_special_tokens_to_tokenizer(tokenizer, model)

Loading checkpoint shards: 100%|██████████| 4/4 [00:09<00:00,  2.31s/it]


In [12]:
import math

MAX_LEN = 10240
SAFETY_BUFFER = 20 # for (part i/N)
SPECIAL_TOKENS = 3  # </p4> + EOS, <p4>

def build_prompt_ids(annotation, idx=None, total=None):
    user_txt = f"In P4 {annotation}"
    if idx is not None:
        user_txt += f" (part {idx}/{total})"
    # tokenize directly
    msgs = [
        {"role":"system", 
         "content": 
         """You are a P4 code generator. 
         Respond with one compilable P4_16 program wrapped in <p4>...</p4> tags. 
         Do not include any explanation or comments. 
         Always include headers, parser, ingress/egress controls, deparser, and main block (e.g., V1Switch(...)). 
         Target BMv2 with v1model and ensure the output works with p4c."""},
        {"role":"user", "content": user_txt}
    ]
    prompt_ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_special_tokens=False, add_generation_prompt=True)
    return prompt_ids

def split_code_toks(code_toks, max_resp):
    n_parts = math.ceil(len(code_toks) / max_resp)
    for i in range(n_parts):
        yield i+1, n_parts, code_toks[i*max_resp:(i+1)*max_resp]

P4_START_TOKEN_ID = tokenizer.convert_tokens_to_ids("<p4>")
P4_END_TOKEN_ID = tokenizer.convert_tokens_to_ids("</p4>")
EOS = tokenizer.eos_token_id

def preprocess(examples):
    out = {"input_ids": [], "attention_mask": [], "loss_computation_start_index": []}
    for code_str, annotation in zip(examples["cleaned_p4"], examples["annotation"]):

        code_toks = tokenizer(code_str, add_special_tokens=False)["input_ids"]
        # first, get minimal overhead (no part tag)
        base_ids = build_prompt_ids(annotation)
        max_resp_base = MAX_LEN - len(base_ids) - SAFETY_BUFFER - SPECIAL_TOKENS

        for idx, total, chunk in split_code_toks(code_toks, max_resp_base):
            prompt_ids = build_prompt_ids(annotation, idx, total)
            # max_resp = MAX_LEN - len(prompt_ids) - SPECIAL_TOKENS
            # # re‑slice chunk in case buffer changed
            # chunk = chunk[:max_resp]

            ids = prompt_ids + [P4_START_TOKEN_ID] + chunk + [P4_END_TOKEN_ID , EOS]
            out["input_ids"].append(ids)
            out["attention_mask"].append([1]*len(ids))
            out["loss_computation_start_index"].append(len(prompt_ids))

    return out

def generate_prompt_and_tokenize(dataset, tokenizer):
  dataset_columns = dataset["train"].column_names

  return dataset.map(
        preprocess,
        batched=True,
        remove_columns=dataset_columns,
        num_proc=os.cpu_count(),
  )


In [13]:
formatted_tokenized_ds = generate_prompt_and_tokenize(ds, tokenizer)

num_proc must be <= 50. Reducing num_proc to 50 for dataset of size 50.
num_proc must be <= 51. Reducing num_proc to 51 for dataset of size 51.


In [14]:
formatted_tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'loss_computation_start_index'],
        num_rows: 383
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'loss_computation_start_index'],
        num_rows: 77
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'loss_computation_start_index'],
        num_rows: 54
    })
})

In [15]:
from peft import PeftModel
is_peft = isinstance(model, PeftModel)

if not hasattr(model, "peft_config"):
  model = prepare_model_for_qlora(model)
else:
  print("You already applied lora!")
  # ModelLoader.delete_instance()

trainable params: 4,423,680 || all params: 15,701,208,576 || trainable%: 0.0282


In [16]:
def get_gradient_norm(model, norm_type=2):
    total_norm = 0.0
    parameters = [p for p in model.parameters() if p.grad is not None]

    if len(parameters) == 0:
        tqdm.write("No gradients found.")
        return

    for p in parameters:
        param_norm = p.grad.data.norm(norm_type)
        total_norm += param_norm.item() ** norm_type

    total_norm = total_norm ** (1. / norm_type)
    return total_norm

In [17]:
from tqdm import tqdm
from torch.amp import autocast

def compute_validation_loss(loader):
    model.eval()
    val_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in tqdm(loader):
            batch = {k: v.to("cuda") for k, v in batch.items()}

            # generate labels:
            batch["labels"] = batch["input_ids"].masked_fill(batch["attention_mask"] == 0, -100)

            start_indices = batch["loss_computation_start_index"]

            if tokenizer.padding_side == "left":
              print("LEFT PADDED!")
              pad_count = (batch["attention_mask"] == 0).sum(dim=1)
              start_indices = start_indices + pad_count

            for i, loss_computation_start_idx in enumerate(start_indices):
              batch["labels"][i, :loss_computation_start_idx] = -100

            # DEBUG 1: check attn mask:
            # ids = batch["input_ids"][1][batch["attention_mask"][1] == 1]
            # print(tokenizer.decode(ids, skip_special_tokens=False))
            # return

            # DEBUD 2: CHECK LABELS SET CORRECTLY
            # print(batch)
            # ids = batch["input_ids"][0][batch["labels"][0] != -100]
            # print(tokenizer.decode(ids, skip_special_tokens=False))
            # return

            batch.pop("loss_computation_start_index")
            # return
            # print(batch)

            with autocast("cuda", dtype=torch.bfloat16):
                loss = model(**batch).loss

            # DEBUG 3
            # print(loss.item())
            # return

            val_loss += loss.item()
            # print(val_loss)
            num_batches += 1

    return val_loss / num_batches if num_batches > 0 else float("inf")

In [18]:
compute_validation_loss(prepare_dataloaders(formatted_tokenized_ds, tokenizer, 4)["test"])

100%|██████████| 14/14 [00:37<00:00,  2.71s/it]


0.6163544335535595

In [19]:
compute_validation_loss(prepare_dataloaders(formatted_tokenized_ds, tokenizer, 4)["validation"])

100%|██████████| 20/20 [00:46<00:00,  2.31s/it]


0.4250662658363581

In [20]:
# deepseek
def generate_from_model(model, tokenizer, instruction):
  model.eval()
  messages = [
      {"role": "system", "content": """You are a P4 code generator. 
         Respond with one compilable P4_16 program wrapped in <p4>...</p4> tags. 
         Do not include any explanation or comments. 
         Always include headers, parser, ingress/egress controls, deparser, and main block (e.g., V1Switch(...)). 
         Target BMv2 with v1model and ensure the output works with p4c."""},
      {"role": "user", "content":   instruction}
  ]
  prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  prompt += " <p4>"
  # DEBUG: UNCOMMET, MAKE SURE PROMPT IS STRUCTURED WELL!
  # print(prompt)
  # return 
  input_ids = tokenizer(prompt , return_tensors="pt").input_ids.to(model.device)

  with torch.no_grad():
      output_ids = model.generate(
          input_ids=input_ids,
          max_new_tokens=512,
          # do_sample=True,
          temperature=0.4,
          pad_token_id=tokenizer.eos_token_id,
          eos_token_id=tokenizer.eos_token_id,
          use_cache=True,


      )

  full_output = tokenizer.decode(output_ids[0], skip_special_tokens=False)
  prompt_text = tokenizer.decode(input_ids[0], skip_special_tokens=False)
  generated = full_output[len(prompt_text):].strip()

  print(f"\n=== INSTRUCTION ===\n{instruction}")
  print(f"\n=== GENERATED RESPONSE ===\n{generated}\n")

In [21]:
generate_from_model(model, tokenizer, "In P4 Implement IPv4 multicast routing and L2/L3 switching with checksum verification")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.



=== INSTRUCTION ===
In P4 Implement IPv4 multicast routing and L2/L3 switching with checksum verification

=== GENERATED RESPONSE ===
...
#include <core.p4>
#include <v1model.p4>

header IPv4_Header {
    bit<9> version;
    bit<4> ihl;
    bit<8> diffserv;
    bit<16> totalLen;
    bit<16> identification;
    bit<3> flags;
    bit<13> fragOffset;
    bit<8> ttl;
    bit<8> protocol;
    bit<32> srcAddr;
    bit<32> dstAddr;
}

header IPv4_Option {
    // Define the fields for IPv4 options if needed
}

header Ethernet_Header {
    bit<48> dstMac;
    bit<48> srcMac;
    bit<16> etherType;
}

parser MyParser(packet_in packet, out headers_t hdr) {
    state start {
        packet.extract(hdr.ethernet);
        transition select(hdr.ethernet.etherType) {
            ETHERTYPE_IP: next(parse_ip);
            default: accept;
        }
    }

    state parse_ip(packet_in packet, inout headers_t hdr) {
        packet.extract(hdr.ipv4);
        transition select(hdr.ipv4.protocol) {
        

In [ ]:
import torch
from tqdm import tqdm
from torch.amp import autocast
import time

EPOCHS = 5
FULL_BATCH_SIZE = 64

# set this higher if you have more VRAM (On 96 GiB GPU GH200, 32 works)
# aim for nicer numbers (multiples of 2,4,8,.. and/or powers of 2) to improve training speed
# 8 should work for 40GB VRAM.
LOCAL_BATCH_SIZE = 4
assert FULL_BATCH_SIZE % LOCAL_BATCH_SIZE == 0

split_dataloaders_dict = prepare_dataloaders(formatted_tokenized_ds, tokenizer, LOCAL_BATCH_SIZE)

optimizer = torch.optim.AdamW(model.parameters(), 8e-5, fused=True)

# print("MODEL VALID LOSS BEFORE FINE-TUNING:")
# model.eval()
# with torch.no_grad():
#     val_loss = compute_validation_loss(split_dataloaders_dict["test"])
#     print(f"Validation loss before ft: {val_loss}")

grad_accum_steps_done = 0
grad_accum_loss = 0.0
optimizer_steps = 0

for epoch in range(EPOCHS):
    model.train()

    for batch in tqdm(split_dataloaders_dict["train"]):

        batch = {k: v.to("cuda") for k, v in batch.items()}

        batch["labels"] = batch["input_ids"].masked_fill(batch["attention_mask"] == 0, -100)

        start_indices = batch["loss_computation_start_index"]

        if tokenizer.padding_side == "left":
            pad_count = (batch["attention_mask"] == 0).sum(dim=1)
            start_indices = start_indices + pad_count

        for i, loss_computation_start_idx in enumerate(start_indices):
            batch["labels"][i, :loss_computation_start_idx] = -100
        
        batch.pop("loss_computation_start_index")

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            loss = (LOCAL_BATCH_SIZE / FULL_BATCH_SIZE) * model(**batch).loss
            loss.backward()

        grad_accum_loss += loss.item()
        grad_accum_steps_done += 1


        if grad_accum_steps_done % (FULL_BATCH_SIZE // LOCAL_BATCH_SIZE) == 0:
          optimizer.step()
          optimizer_steps += 1

          print(f"||∇f_w|| : {get_gradient_norm(model)} | full_batch_loss: {grad_accum_loss} ")
          optimizer.zero_grad(set_to_none=True)
          grad_accum_loss = 0.0

    print("\n GENERATING: \n")
    model.eval()
    with torch.no_grad():
        val_loss = compute_validation_loss(split_dataloaders_dict["validation"])
        print("-"* 20)
        print(f"Validation loss on epoch {epoch + 1}: {val_loss}")
        print("-"* 20)
        
        generate_from_model(model, tokenizer, "Implement an IPv4 router that drops packets if TTL ≤ 1, and otherwise decrements TTL and forwards using LPM on the destination IP address. ")


    # model.save_pretrained(f"lora_checkpoint_e{epoch + 1}/")
    # tokenizer.save_pretrained(f"lora_checkpoint_e{epoch+1}/")


  0%|          | 0/96 [00:00<?, ?it/s]/usr/lib/python3/dist-packages/torch/_dynamo/eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
 17%|█▋        | 16/96 [02:12<10:40,  8.01s/it]

||∇f_w|| : 0.5344058886599947 | full_batch_loss: 0.6676695365458727 


 33%|███▎      | 32/96 [05:00<11:11, 10.49s/it]

||∇f_w|| : 0.41190897154783657 | full_batch_loss: 0.5641203690320253 


 50%|█████     | 48/96 [07:49<09:59, 12.48s/it]

||∇f_w|| : 0.4382259206815338 | full_batch_loss: 0.5764430277049541 


 57%|█████▋    | 55/96 [09:02<06:25,  9.40s/it]

In [ ]:
# from peft import PeftModel

# # Save only LoRA adapter weights
# model.save_pretrained("lora_checkpoint/")
# tokenizer.save_pretrained("lora_checkpoint/")